# Example 15.6 / 1: Structure the Problem, Then Refine with Solver Errors

Companion notebook for **15.6 Talks on AI for Optimization** in the AMPL lab Documentation, section "Structure the Problem, Then Refine with Solver Errors". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Structure the Problem, Then Refine with Solver Errors

Paste the exact error back into the conversation, with the model. The corrected constraint uses
`stock`, and the model then solves:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                            # products
set COMP;                            # components
param price {PROD} > 0;              # revenue per unit
param need {COMP, PROD} >= 0;        # components per unit of product
param stock {COMP} >= 0;             # components on hand

var Make {PROD} >= 0;                # units produced

maximize Revenue: sum {p in PROD} price[p] * Make[p];  # total revenue

# components used fit within stock
subject to Inventory {c in COMP}: sum {p in PROD} need[c,p] * Make[p] <= stock[c];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["A", "B"]
ampl.set["COMP"] = ["X1", "X2"]
ampl.param["price"] = {"A": 3, "B": 4}
ampl.param["need"] = {("X1", "A"): 1, ("X1", "B"): 2, ("X2", "A"): 3, ("X2", "B"): 1}
ampl.param["stock"] = {"X1": 10, "X2": 15}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result, ampl.obj["Revenue"].value(), ampl.var["Make"].to_dict())

HiGHS 1.14.0: optimal solution; objective 24
2 simplex iterations
0 barrier iterations
solved 24.0 {'A': 4, 'B': 3}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := A B;
set COMP := X1 X2;
param price := A 3  B 4;
param need:  A  B :=
X1           1  2
X2           3  1;
param stock := X1 10  X2 15;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Revenue, Make;

HiGHS 1.14.0: optimal solution; objective 24
2 simplex iterations
0 barrier iterations
Revenue = 24

Make [*] :=
A  4
B  3
;

